Google Colab Link: `https://colab.research.google.com/drive/1dtaGivk3QBTwZHYgpqLFtx2IMjGI65Oc?usp=sharing`

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import os
import requests
import json
from pathlib import Path
from datetime import datetime, timezone

import xgboost as xgb

In [2]:
MODEL_PATH  = '/content/model_data.pkl'

OWM_API_KEY = 'API_KEY'
OWM_UNITS   = 'metric'

REGIONS = {
    'Jakarta Pusat'   : (-6.1862, 106.8063),
    'Jakarta Selatan' : (-6.2615, 106.8106),
    'Jakarta Timur'   : (-6.2251, 106.9004),
    'Jakarta Utara'   : (-6.1335, 106.8821),
}

FLOOD_THRESHOLD = 0.45

In [3]:
model_data = None

with open(MODEL_PATH, 'rb') as f:
    model_data = pickle.load(f)

le_region   = model_data['le_region']
region_map  = model_data['region_map']
FEATURE_COL = model_data['feature_cols']
scaler      = model_data['scaler']
model       = model_data['flood_model']
best_params = model_data['best_params']

In [4]:
def fetch_owm_forecast(region_name, lat, lon):
    url    = 'https://api.openweathermap.org/data/2.5/forecast'
    params = {
        'lat'   : lat,
        'lon'   : lon,
        'appid' : OWM_API_KEY,
        'units' : OWM_UNITS,
        'cnt'   : 40,
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()

    data = resp.json()
    data['_region_name'] = region_name

    return data

raw_responses = {}
for region, (lat, lon) in REGIONS.items():
    raw_responses[region] = fetch_owm_forecast(region, lat, lon)
    print(f'{region:15} : OK  ({len(raw_responses[region]["list"])} slots)')

Jakarta Pusat   : OK  (40 slots)
Jakarta Selatan : OK  (40 slots)
Jakarta Timur   : OK  (40 slots)
Jakarta Utara   : OK  (40 slots)


In [5]:
def parse_owm_response(data):
    rows = []
    region_name = data.get('_region_name', 'Unknown')

    for slot in data['list']:
        rows.append({
            'region_name': region_name,
            'dt_txt'     : slot['dt_txt'],
            'temp_min'   : slot['main']['temp_min'],
            'temp_max'   : slot['main']['temp_max'],
            'temp'       : slot['main']['temp'],
            'humidity'   : slot['main']['humidity'],
            'rain_3h'    : slot.get('rain', {}).get('3h', 0.0),
            'wind_speed' : slot['wind']['speed'],
            'wind_deg'   : slot['wind']['deg'],
        })

    df = pd.DataFrame(rows)
    df['dt_txt'] = pd.to_datetime(df['dt_txt'])
    return df

frames_3h = [parse_owm_response(v) for v in raw_responses.values()]
df_3h     = pd.concat(frames_3h, ignore_index=True)

df_3h.head()

,region_name,dt_txt,temp_min,temp_max,temp,humidity,rain_3h,wind_speed,wind_deg
0,Jakarta Pusat,2026-06-06 09:00:00,34.42,34.53,34.53,46,0.00,6.04,61
1,Jakarta Pusat,2026-06-06 12:00:00,31.20,33.42,33.42,53,0.00,5.57,76
2,Jakarta Pusat,2026-06-06 15:00:00,30.49,31.84,31.84,66,0.11,4.84,96
3,Jakarta Pusat,2026-06-06 18:00:00,30.46,30.46,30.46,72,0.12,4.50,102
4,Jakarta Pusat,2026-06-06 21:00:00,30.20,30.20,30.20,73,0.00,4.16,103


In [6]:
def circular_mean_deg(angles_deg):
    rad = np.deg2rad(angles_deg)
    return np.degrees(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())) % 360

def preprocess_inference(df):
    df_3h = df.copy()

    df_3h['date'] = df_3h['dt_txt'].dt.normalize()

    df_daily = (
        df_3h
        .groupby(['region_name', 'date'], sort=True)
        .agg(
            Tn     = ('temp_min', 'min'),
            Tx     = ('temp_max', 'max'),
            Tavg   = ('temp', 'mean'),
            RH_avg = ('humidity', 'mean'),
            RR     = ('rain_3h', 'sum'),
            ff_avg = ('wind_speed', 'mean'),
            ddd_x  = ('wind_deg', circular_mean_deg),
        ).reset_index()
    )

    df_daily = df_daily.sort_values(['region_name', 'date']).reset_index(drop=True)
    df_daily['region_id'] = le_region.transform(df_daily['region_name'].astype(str))

    rad = np.deg2rad(df_daily['ddd_x'])
    df_daily['wind_direction_sin'] = np.sin(rad)
    df_daily['wind_direction_cos'] = np.cos(rad)
    df_daily.drop(columns=['ddd_x'], inplace=True)

    return df_daily


df_daily = preprocess_inference(df_3h)
df_daily.head()

,region_name,date,Tn,Tx,Tavg,RH_avg,RR,ff_avg,region_id,wind_direction_sin,wind_direction_cos
0,Jakarta Pusat,2026-06-06,30.20,34.53,32.09000,62.000,0.23,5.02200,0,0.999220,0.039481
1,Jakarta Pusat,2026-06-07,28.64,34.77,31.36500,62.875,0.00,4.03625,0,0.996302,0.085919
2,Jakarta Pusat,2026-06-08,27.24,31.90,29.29000,63.750,0.00,3.05875,0,0.965842,0.259133
3,Jakarta Pusat,2026-06-09,27.50,32.68,29.56500,60.500,0.11,3.04500,0,0.984407,0.175906
4,Jakarta Pusat,2026-06-10,27.42,33.02,29.59375,61.500,0.65,3.22250,0,0.949375,0.314146


In [7]:
def feature_engineer_inference(df):
    df = df.copy().sort_values(['region_id', 'date']).reset_index(drop=True)

    df['month']       = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['month_sin']   = np.sin(2 * np.pi * df['month']       / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month']       / 12)
    df['doy_sin']     = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['doy_cos']     = np.cos(2 * np.pi * df['day_of_year'] / 365)
    df.drop(columns=['month', 'day_of_year'], inplace=True)

    grp = df.groupby('region_id')['RR']
    df['rainfall_lag1d'] = grp.shift(1).fillna(0.0)
    df['rainfall_lag3d'] = grp.shift(3).fillna(0.0)
    df['rainfall_lag7d'] = grp.shift(7).fillna(0.0)

    df['rainfall_rolling3d_sum']  = grp.transform(lambda s: s.rolling(3,  min_periods=1).sum())
    df['rainfall_rolling7d_sum']  = grp.transform(lambda s: s.rolling(7,  min_periods=1).sum())
    df['rainfall_rolling14d_sum'] = grp.transform(lambda s: s.rolling(14, min_periods=1).sum())
    df['rainfall_rolling3d_max']  = grp.transform(lambda s: s.rolling(3,  min_periods=1).max())
    df['rainfall_rolling7d_max']  = grp.transform(lambda s: s.rolling(7,  min_periods=1).max())
    df['rainfall_rolling7d_std']  = grp.transform(lambda s: s.rolling(7, min_periods=2).std().fillna(0.0))

    grp_rh = df.groupby('region_id')['RH_avg']
    df['humidity_rolling3d_mean'] = grp_rh.transform(lambda s: s.rolling(3, min_periods=1).mean())
    df['humidity_rolling7d_mean'] = grp_rh.transform(lambda s: s.rolling(7, min_periods=1).mean())

    rain_cols = [c for c in df.columns if 'rainfall' in c]
    for col in rain_cols:
        df[col] = np.log1p(df[col])

    return df

df_feat = feature_engineer_inference(df_daily)
df_feat.head()

,region_name,date,Tn,Tx,Tavg,RH_avg,RR,ff_avg,region_id,wind_direction_sin,...,rainfall_lag3d,rainfall_lag7d,rainfall_rolling3d_sum,rainfall_rolling7d_sum,rainfall_rolling14d_sum,rainfall_rolling3d_max,rainfall_rolling7d_max,rainfall_rolling7d_std,humidity_rolling3d_mean,humidity_rolling7d_mean
0,Jakarta Pusat,2026-06-06,30.20,34.53,32.09000,62.000,0.23,5.02200,0,0.999220,...,0.000000,0.0,0.207014,0.207014,0.207014,0.207014,0.207014,0.000000,62.000000,62.00000
1,Jakarta Pusat,2026-06-07,28.64,34.77,31.36500,62.875,0.00,4.03625,0,0.996302,...,0.000000,0.0,0.207014,0.207014,0.207014,0.207014,0.207014,0.150689,62.437500,62.43750
2,Jakarta Pusat,2026-06-08,27.24,31.90,29.29000,63.750,0.00,3.05875,0,0.965842,...,0.000000,0.0,0.207014,0.207014,0.207014,0.207014,0.207014,0.124684,62.875000,62.87500
3,Jakarta Pusat,2026-06-09,27.50,32.68,29.56500,60.500,0.11,3.04500,0,0.984407,...,0.207014,0.0,0.104360,0.292670,0.292670,0.104360,0.207014,0.104087,62.375000,62.28125
4,Jakarta Pusat,2026-06-10,27.42,33.02,29.59375,61.500,0.65,3.22250,0,0.949375,...,0.000000,0.0,0.565314,0.688135,0.688135,0.500775,0.500775,0.238973,61.916667,62.12500


In [8]:
X_infer = df_feat[FEATURE_COL].values
X_infer_scaled = scaler.transform(X_infer)

flood_proba = model.predict_proba(X_infer_scaled)[:, 1]
flood_pred  = (flood_proba >= FLOOD_THRESHOLD).astype(int)

df_result = df_feat[['region_name', 'date', 'Tn', 'Tx', 'Tavg', 'RH_avg', 'RR']].copy()
df_result['flood_probability'] = flood_proba.round(4)
df_result['flood_alert']       = flood_pred
df_result['alert_label']       = df_result['flood_alert'].map({0: 'No Flood', 1: '⚠ FLOOD'})

df_result.head()

,region_name,date,Tn,Tx,Tavg,RH_avg,RR,flood_probability,flood_alert,alert_label
0,Jakarta Pusat,2026-06-06,30.20,34.53,32.09000,62.000,0.23,0.0133,0,No Flood
1,Jakarta Pusat,2026-06-07,28.64,34.77,31.36500,62.875,0.00,0.0103,0,No Flood
2,Jakarta Pusat,2026-06-08,27.24,31.90,29.29000,63.750,0.00,0.0085,0,No Flood
3,Jakarta Pusat,2026-06-09,27.50,32.68,29.56500,60.500,0.11,0.0076,0,No Flood
4,Jakarta Pusat,2026-06-10,27.42,33.02,29.59375,61.500,0.65,0.0097,0,No Flood


In [9]:
def print_summary(df_result):
    summary = df_result[['region_name', 'date', 'RR', 'RH_avg', 'Tavg', 'flood_probability', 'alert_label']].copy()
    summary.columns = ['Region', 'Date', 'Rain (mm)', 'Humidity (%)', 'Temp (°C)', 'Flood Prob.', 'Alert']
    return summary

print_summary(df_result)

,Region,Date,Rain (mm),Humidity (%),Temp (°C),Flood Prob.,Alert
0,Jakarta Pusat,2026-06-06,0.23,62.000000,32.090000,0.0133,No Flood
1,Jakarta Pusat,2026-06-07,0.00,62.875000,31.365000,0.0103,No Flood
2,Jakarta Pusat,2026-06-08,0.00,63.750000,29.290000,0.0085,No Flood
3,Jakarta Pusat,2026-06-09,0.11,60.500000,29.565000,0.0076,No Flood
4,Jakarta Pusat,2026-06-10,0.65,61.500000,29.593750,0.0097,No Flood
5,Jakarta Pusat,2026-06-11,0.36,61.666667,29.663333,0.0075,No Flood
6,Jakarta Selatan,2026-06-06,0.25,61.000000,32.088000,0.0420,No Flood
7,Jakarta Selatan,2026-06-07,0.00,62.000000,31.333750,0.0270,No Flood
8,Jakarta Selatan,2026-06-08,0.00,62.375000,29.357500,0.0246,No Flood
9,Jakarta Selatan,2026-06-09,0.00,58.750000,29.621250,0.0225,No Flood
